# `financialtools.processor` — User Guide

Core data-acquisition and metric-evaluation module: fetches raw financials from Yahoo Finance,
reshapes them into long-format DataFrames, computes 11 fundamental metrics, scores them on a
1–5 scale, and detects red flags.

```
financialtools/processor.py
│
├── RateLimiter(per_minute, per_hour, per_day)
│   └── acquire()               block until a call slot is available
│
├── Downloader(ticker)
│   ├── from_ticker(ticker)     classmethod — fetch + reshape; never raises
│   ├── get_merged_data()       long-format DataFrame (balance sheet + income + cashflow)
│   ├── get_info_data()         DataFrame with marketCap, forwardPE, etc.
│   ├── combine_merged_data()   classmethod — concat across multiple Downloader instances
│   ├── combine_info_data()     classmethod — concat info DataFrames
│   └── stream_download()       classmethod — yield Downloaders one by one, save to Parquet
│
├── FundamentalTraderAssistant(data, weights)
│   ├── evaluate()              full pipeline → dict with 5 keys
│   ├── compute_metrics()       compute SCORED_METRICS columns
│   ├── compute_valuation_metrics()  P/E, P/B, P/FCF, EarningsYield, FCFYield
│   ├── compute_scores()        score each metric 1–5
│   ├── raw_red_flags()         cash-flow quality flags
│   └── metrics_red_flags()     threshold-based flags
│
└── Module-level helpers
    ├── SCORED_METRICS          list[str] — 11 metric column names
    ├── _EMPTY_RESULT_KEYS      tuple[str] — canonical evaluate() output keys
    └── _empty_result()         factory → {k: pd.DataFrame() for k in _EMPTY_RESULT_KEYS}
```

| Symbol | Needs yfinance | Needs LLM | Best for |
|---|---|---|---|
| `RateLimiter` | no | no | throttling any external API call loop |
| `Downloader.from_ticker` | **yes** | no | fetching one ticker's financials |
| `Downloader.get_merged_data` | no | no | accessing already-fetched data |
| `FundamentalTraderAssistant` | no | no | scoring metrics from any merged DataFrame |
| `_empty_result` | no | no | safe fallback / test scaffolding |

**Prerequisites:**
- Sections 1–5 and 7–9 run with **no API key** and **no internet connection**.
- Section 6 (Live Downloader demo) requires `yfinance` and an active internet connection.

## Sections
1. [Setup](#1-setup)
2. [Module-level constants](#2-module-level-constants)
3. [RateLimiter](#3-ratelimiter)
4. [FundamentalTraderAssistant — construction](#4-fundamentaltraderassistant--construction)
5. [FundamentalTraderAssistant — evaluate()](#5-fundamentaltraderassistant--evaluate)
6. [Downloader (Live)](#6-downloader-live)
7. [Error handling](#7-error-handling)
8. [Common failure modes](#8-common-failure-modes)
9. [End-to-end pattern](#9-end-to-end-pattern)

## 1 — Setup

In [ ]:
import logging
import numpy as np
import pandas as pd

from financialtools.processor import (
    RateLimiter,
    Downloader,
    FundamentalTraderAssistant,
    SCORED_METRICS,
    _EMPTY_RESULT_KEYS,
    _empty_result,
)
from financialtools.exceptions import EvaluationError

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    force=True,
)
print("Imports OK")

## 2 — Module-level constants

`SCORED_METRICS` is a reference list of the 11 metric column names produced by
`compute_metrics()`. It is kept for documentation and test-scaffolding purposes.
`evaluate()` and `compute_scores()` **derive `value_vars` dynamically** from
`compute_metrics()` output columns (all columns that are not `ticker`, `time`, or `sector`),
so adding a new metric to `compute_metrics()` automatically includes it in scoring without
editing `SCORED_METRICS`.

`_EMPTY_RESULT_KEYS` is the tuple of keys that every `evaluate()` return value must
contain. `_empty_result()` is the factory that creates a fresh dict of empty DataFrames
with exactly those keys.

> Always call `_empty_result()` — never copy `_EMPTY_RESULT` directly — because a shallow
> copy would share DataFrame objects across calls.

In [ ]:
print(f"SCORED_METRICS ({len(SCORED_METRICS)} items):")
for m in SCORED_METRICS:
    print(f"  {m}")

print(f"\n_EMPTY_RESULT_KEYS : {_EMPTY_RESULT_KEYS}")

er = _empty_result()
print(f"\n_empty_result() keys  : {list(er.keys())}")
print(f"All values empty      : {all(v.empty for v in er.values())}")
print(f"Distinct objects      : {len({id(v) for v in er.values()})} (should equal {len(er)})")

## 3 — RateLimiter

`RateLimiter` is a thread-safe sliding-window rate limiter. It tracks call timestamps and
sleeps the calling thread until all three windows (minute, hour, day) have a free slot.

### Constructor

```python
RateLimiter(per_minute=60, per_hour=360, per_day=8000)
```

| Parameter | Default | Meaning |
|---|---|---|
| `per_minute` | 60 | max calls within any 60-second window |
| `per_hour` | 360 | max calls within any 3600-second window |
| `per_day` | 8000 | max calls within any 86400-second window |

### acquire()

Blocks until all three windows have a free slot, then records the timestamp. The internal
`calls` list is pruned to the last 24 hours on every `acquire()` call to bound memory usage.

### Thread safety

All state mutations (reading + writing `self.calls`) are protected by `threading.Lock`.
It is safe to share one `RateLimiter` instance across threads — for example across a
`ThreadPoolExecutor` that downloads multiple tickers in parallel.

### Typical usage

Pass a `RateLimiter` instance to `Downloader.stream_download()` to pace bulk downloads.

In [ ]:
import time
import threading

# --- Basic construction ---
limiter = RateLimiter(per_minute=10, per_hour=100, per_day=1000)
print(f"per_minute : {limiter.per_minute}")
print(f"per_hour   : {limiter.per_hour}")
print(f"per_day    : {limiter.per_day}")
print(f"calls so far: {len(limiter.calls)}")

# --- acquire() under the limit — should return immediately ---
for i in range(3):
    limiter.acquire()
    print(f"  call {i+1}: acquired at {time.strftime('%H:%M:%S')}")

print(f"calls recorded: {len(limiter.calls)}")

In [ ]:
# --- Thread-safety demo: 5 threads sharing one RateLimiter with per_minute=5 ---
# Each thread calls acquire() once. With per_minute=5, the first 5 calls go through
# immediately; a 6th would block. Here we fire exactly 5 to show safe concurrent use.

results = []
lock = threading.Lock()

def worker(tid):
    limiter_shared.acquire()
    with lock:
        results.append(f"thread-{tid} acquired at {time.strftime('%H:%M:%S')}")

limiter_shared = RateLimiter(per_minute=5, per_hour=100, per_day=1000)
threads = [threading.Thread(target=worker, args=(i,)) for i in range(5)]
for t in threads:
    t.start()
for t in threads:
    t.join()

for r in sorted(results):
    print(r)
print(f"Total calls recorded: {len(limiter_shared.calls)}")

## 4 — FundamentalTraderAssistant — construction

`FundamentalTraderAssistant` validates its inputs eagerly in `__init__` and raises
`EvaluationError` for any of three guard conditions:

| Guard | Raises when |
|---|---|
| Empty ticker | `data['ticker'].dropna().unique()` is empty |
| Multiple tickers | `data` contains more than one distinct non-null ticker value |
| Empty sector in weights | `weights['sector'].dropna().unique()` is empty |
| Multiple sectors in weights | `weights` contains more than one distinct sector |

All three attributes start as empty DataFrames:

```python
self.metrics       = pd.DataFrame()
self.eval_metrics  = pd.DataFrame()
self.scores        = pd.DataFrame()
```

They are populated lazily by `compute_metrics()`, `compute_valuation_metrics()`, and
`evaluate()`.

### Mock data for offline demos

The merged DataFrame produced by `Downloader.get_merged_data()` has one row per
`(ticker, time)` pair and one column per financial field. The cells below use a minimal
mock with the columns required by `compute_metrics()` and `compute_valuation_metrics()`.

In [ ]:
rng = np.random.default_rng(seed=42)

# Build a mock merged DataFrame for a single ticker, four annual periods.
TICKER = "MOCK"
YEARS  = ["2020", "2021", "2022", "2023"]
N      = len(YEARS)

mock_data = pd.DataFrame({
    "ticker"                       : TICKER,
    "time"                         : YEARS,
    # Income statement
    "total_revenue"                : rng.uniform(8e9,  12e9,  N),
    "gross_profit"                 : rng.uniform(4e9,   6e9,  N),
    "operating_income"             : rng.uniform(1e9,   3e9,  N),
    "net_income_common_stockholders": rng.uniform(0.5e9, 2e9, N),
    "ebitda"                       : rng.uniform(2e9,   4e9,  N),
    "diluted_eps"                  : rng.uniform(1.5,   6.0,  N),
    # Balance sheet
    "total_assets"                 : rng.uniform(20e9, 40e9,  N),
    "common_stock_equity"          : rng.uniform(5e9,  15e9,  N),
    "total_debt"                   : rng.uniform(3e9,   8e9,  N),
    "current_assets"               : rng.uniform(5e9,  10e9,  N),
    "current_liabilities"          : rng.uniform(3e9,   6e9,  N),
    "sharesoutstanding"            : rng.uniform(1e9,   2e9,  N),
    # Cash flow
    "free_cash_flow"               : rng.uniform(0.8e9, 3e9,  N),
    "operating_cash_flow"          : rng.uniform(1e9,   4e9,  N),
    # Market data (usually joined from info)
    "marketcap"                    : rng.uniform(30e9,  80e9, N),
    "currentprice"                 : rng.uniform(100,   400,  N),
})

print(f"mock_data shape : {mock_data.shape}")
mock_data[["ticker", "time", "total_revenue", "gross_profit", "free_cash_flow"]].head()

In [ ]:
# Mock weights DataFrame — one row per scored metric, single sector.
mock_weights = pd.DataFrame({
    "sector" : "Technology",
    "metrics": SCORED_METRICS,
    "weights": [1.5, 1.2, 1.2, 1.0, 1.3, 1.3, 1.1, 1.1, 1.0, 1.2, 1.1],
})

print(f"mock_weights shape : {mock_weights.shape}")
print(mock_weights.to_string(index=False))

# Construct the assistant — should succeed with one ticker + one sector.
fta = FundamentalTraderAssistant(data=mock_data, weights=mock_weights)
print(f"\nfta.ticker  : {fta.ticker}")
print(f"fta.sector  : {fta.sector}")
print(f"fta.metrics (before evaluate) empty : {fta.metrics.empty}")
print(f"fta.scores  (before evaluate) empty : {fta.scores.empty}")

## 5 — FundamentalTraderAssistant — evaluate()

`evaluate()` runs the full scoring pipeline and returns a dict with five keys:

| Key | Type | Content |
|---|---|---|
| `metrics` | `pd.DataFrame` | wide — one row per `(ticker, time)`, one column per metric in `SCORED_METRICS` |
| `eval_metrics` | `pd.DataFrame` | wide — valuation metrics: P/E, P/B, P/FCF, EarningsYield, FCFYield |
| `composite_scores` | `pd.DataFrame` | one row per `(sector, ticker, time)` with `composite_score` |
| `raw_red_flags` | `pd.DataFrame` | cash-flow quality flags (negative FCF/OCF, EBITDA >> OCF) |
| `red_flags` | `pd.DataFrame` | threshold-based flags (negative margins, high D/E, etc.) |

### Invariant: never raises

`evaluate()` wraps its entire body in `try/except`. On any internal failure it logs the
error and returns `_empty_result()` — a dict of five empty DataFrames — so the caller
always receives the same shape.

### Composite score formula

```
composite_score = sum(score_i * weight_i) / sum(weight_i)
```

where `score_i` is the 1–5 score for metric `i` (from `score_metric()`) and `weight_i`
comes from the `weights` DataFrame passed to the constructor.

In [ ]:
result = fta.evaluate()

print("Result keys:", list(result.keys()))
print(f"All keys present: {set(result.keys()) == set(_EMPTY_RESULT_KEYS)}")

print(f"\n--- metrics ({result['metrics'].shape}) ---")
print(result["metrics"][["ticker", "time", "GrossMargin", "ROE", "DebtToEquity"]].to_string(index=False))

print(f"\n--- eval_metrics ({result['eval_metrics'].shape}) ---")
print(result["eval_metrics"][["ticker", "time", "P/E", "P/B", "FCFYield"]].to_string(index=False))

print(f"\n--- composite_scores ({result['composite_scores'].shape}) ---")
print(result["composite_scores"].to_string(index=False))

In [ ]:
# Inspect red flags (mock data uses positive values so flags may be empty)
raw_rf = result["raw_red_flags"]
rf     = result["red_flags"]

print(f"raw_red_flags rows : {len(raw_rf)}")
if not raw_rf.empty:
    print(raw_rf.to_string(index=False))
else:
    print("  (none — all cash flows positive in this mock)")

print(f"\nred_flags rows     : {len(rf)}")
if not rf.empty:
    print(rf.to_string(index=False))
else:
    print("  (none — all metrics within thresholds in this mock)")

# Force a red-flag scenario by injecting a negative FCF row
mock_bad = mock_data.copy()
mock_bad.loc[0, "free_cash_flow"] = -500_000_000
mock_bad.loc[1, "net_income_common_stockholders"] = -200_000_000

fta_bad = FundamentalTraderAssistant(data=mock_bad, weights=mock_weights)
result_bad = fta_bad.evaluate()

print("\n--- raw_red_flags with negative FCF ---")
print(result_bad["raw_red_flags"].to_string(index=False))

print("\n--- red_flags with negative net margin ---")
print(result_bad["red_flags"].to_string(index=False))